In [36]:
from torchvision.transforms import Compose
from torchvision.transforms import Lambda
from torchvision.transforms import Normalize
from torchvision.transforms import Resize
from torchvision.transforms import ToTensor

import sys
import torch
from PIL import Image
from pathlib import Path

from torch.nn import Sequential, Conv2d, ReLU, Module, MaxPool2d, Linear, Flatten
from torch.nn import BatchNorm2d, Dropout2d, Dropout
from src.data.dataset import create_data_loaders

In [37]:
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

X_MODEL_PATH = PROJECT_ROOT / "04_models" / "base_cnn.pth"
Y_MODEL_PATH = PROJECT_ROOT / "04_models" / "model_BatchNormAndDropoutCNN.pth"

print(Y_MODEL_PATH)

C:\Users\sveto\OneDrive\Desktop\cats_and_dogs_classification\04_models\model_BatchNormAndDropoutCNN.pth


In [38]:
RANDOM_IMAGES_DIR = Path.cwd() / "random_imgs"

print(RANDOM_IMAGES_DIR)

C:\Users\sveto\OneDrive\Desktop\cats_and_dogs_classification\02_notebooks\random_imgs


In [39]:
class BaseCNN(Module):

    def __init__(self):
        super().__init__()

        self.features = Sequential(
            Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),

            Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
        )

        self.classifier = Sequential(
            Flatten(),
            Linear(in_features=64 * 32 * 32, out_features=128),
            ReLU(),
            Linear(in_features=128, out_features=2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [40]:
model = BaseCNN()

model.load_state_dict(
    torch.load(
        X_MODEL_PATH,
        map_location="cpu",
        weights_only=True,
    )
)

model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [41]:
train_loader, validation_loader, test_loader = create_data_loaders()

In [42]:
correct_predictions = 0
total_samples = 0

model.eval()

with torch.no_grad():

    for images, labels in test_loader:
        outputs = model(images)
        predictions = outputs.argmax(dim=1)
        correct_predictions += (predictions == labels).sum().item()
        total_samples += labels.size(0)

test_accuracy = correct_predictions / total_samples

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")

Test accuracy: 0.7929
Test accuracy: 79.29%


In [43]:
prediction_transform = Compose([
    Resize((128, 128)),
    ToTensor(),
    Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5],
    ),
])

In [44]:
def predict_image(model, image_path, transform, class_names):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0)

    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)

        predicted_index = probabilities.argmax(dim=1).item()
        confidence = probabilities[0, predicted_index].item()

    predicted_class = class_names[predicted_index]

    return predicted_class, confidence

In [45]:
class_names = test_loader.dataset.classes

supported_extensions = {".jpg", ".jpeg", ".png"}

for image_path in sorted(RANDOM_IMAGES_DIR.iterdir()):

    if image_path.suffix.lower() not in supported_extensions:
        continue

    predicted_class, confidence = predict_image(
        model=model,
        image_path=image_path,
        transform=prediction_transform,
        class_names=class_names,
    )

    print(
        f"{image_path.name}: "
        f"prediction={predicted_class}, "
        f"confidence={confidence:.2%}"
    )

cat-1.jpg: prediction=cats, confidence=59.06%
cat-10.jpg: prediction=dogs, confidence=52.19%
cat-11.jpg: prediction=dogs, confidence=52.60%
cat-12.jpg: prediction=dogs, confidence=71.99%
cat-13.jpg: prediction=dogs, confidence=79.06%
cat-14.jpg: prediction=cats, confidence=67.20%
cat-15.jpg: prediction=dogs, confidence=70.01%
cat-2.jpg: prediction=dogs, confidence=64.83%
cat-3.jpg: prediction=dogs, confidence=99.49%
cat-4.jpg: prediction=dogs, confidence=61.40%
cat-5.jpg: prediction=dogs, confidence=84.26%
cat-6.jpg: prediction=dogs, confidence=77.77%
cat-7.jpg: prediction=cats, confidence=86.02%
cat-8.jpg: prediction=dogs, confidence=92.99%
cat-9.jpg: prediction=dogs, confidence=51.96%
dog-1.jpg: prediction=dogs, confidence=97.23%
dog-10.jpg: prediction=dogs, confidence=88.87%
dog-11.jpg: prediction=dogs, confidence=81.50%
dog-12.jpg: prediction=dogs, confidence=70.04%
dog-13.jpg: prediction=dogs, confidence=98.34%
dog-14.jpg: prediction=dogs, confidence=88.79%
dog-15.jpg: prediction=

In [46]:
correct_predictions = 0
total_images = 0

for image_path in sorted(RANDOM_IMAGES_DIR.iterdir()):

    if image_path.suffix.lower() not in supported_extensions:
        continue

    predicted_class, confidence = predict_image(
        model=model,
        image_path=image_path,
        transform=prediction_transform,
        class_names=class_names,
    )

    if image_path.stem.startswith("cat"):
        actual_class = "cats"
    elif image_path.stem.startswith("dog"):
        actual_class = "dogs"
    else:
        continue

    is_correct = predicted_class == actual_class

    correct_predictions += int(is_correct)
    total_images += 1

    print(
        f"{image_path.name} | "
        f"Actual: {actual_class} | "
        f"Predicted: {predicted_class} | "
        f"Confidence: {confidence:.2%} | "
        f"Correct: {is_correct}"
    )

external_accuracy = correct_predictions / total_images

print()
print(f"Correct predictions: {correct_predictions}/{total_images}")
print(f"External images accuracy: {external_accuracy:.2%}")

cat-1.jpg | Actual: cats | Predicted: cats | Confidence: 59.06% | Correct: True
cat-10.jpg | Actual: cats | Predicted: dogs | Confidence: 52.19% | Correct: False
cat-11.jpg | Actual: cats | Predicted: dogs | Confidence: 52.60% | Correct: False
cat-12.jpg | Actual: cats | Predicted: dogs | Confidence: 71.99% | Correct: False
cat-13.jpg | Actual: cats | Predicted: dogs | Confidence: 79.06% | Correct: False
cat-14.jpg | Actual: cats | Predicted: cats | Confidence: 67.20% | Correct: True
cat-15.jpg | Actual: cats | Predicted: dogs | Confidence: 70.01% | Correct: False
cat-2.jpg | Actual: cats | Predicted: dogs | Confidence: 64.83% | Correct: False
cat-3.jpg | Actual: cats | Predicted: dogs | Confidence: 99.49% | Correct: False
cat-4.jpg | Actual: cats | Predicted: dogs | Confidence: 61.40% | Correct: False
cat-5.jpg | Actual: cats | Predicted: dogs | Confidence: 84.26% | Correct: False
cat-6.jpg | Actual: cats | Predicted: dogs | Confidence: 77.77% | Correct: False
cat-7.jpg | Actual: cats

### Experiment 2: CNN with Batch Normalization and Dropout
---

In [49]:
class BatchNormAndDropoutCNN(Module):

    def __init__(self):
        super().__init__()

        self.features = Sequential(
            Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            BatchNorm2d(num_features=32),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Dropout2d(p=0.2),

            Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            BatchNorm2d(num_features=64),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Dropout2d(p=0.2),

            Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            BatchNorm2d(num_features=128),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Dropout2d(p=0.3),

            Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            BatchNorm2d(num_features=256),
            ReLU(),
            MaxPool2d(kernel_size=3),
            Dropout2d(p=0.3),
        )

        self.classifier = Sequential(
            Flatten(),
            Linear(in_features=256 * 5 * 5, out_features=128),
            ReLU(),
            Dropout(p=0.5),
            Linear(in_features=128, out_features=2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [50]:
model_BatchNormAndDropoutCNN = BatchNormAndDropoutCNN()

model_BatchNormAndDropoutCNN.load_state_dict(
    torch.load(Y_MODEL_PATH, map_location="cpu")
)

model_BatchNormAndDropoutCNN.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [51]:
train_loader, validation_loader, test_loader = create_data_loaders()

In [52]:
correct_predictions = 0
total_samples = 0

model_BatchNormAndDropoutCNN.eval()

with torch.no_grad():

    for images, labels in test_loader:
        outputs = model_BatchNormAndDropoutCNN(images)
        predictions = outputs.argmax(dim=1)

        correct_predictions += (predictions == labels).sum().item()
        total_samples += labels.size(0)

validation_accuracy = correct_predictions / total_samples

print(f"Validation accuracy: {validation_accuracy:.4f}")
print(f"Validation accuracy: {validation_accuracy:.2%}")

Validation accuracy: 0.8699
Validation accuracy: 86.99%


In [53]:
correct_predictions = 0
total_images = 0

for image_path in sorted(RANDOM_IMAGES_DIR.iterdir()):

    if image_path.suffix.lower() not in supported_extensions:
        continue

    predicted_class, confidence = predict_image(
        model=model_BatchNormAndDropoutCNN,
        image_path=image_path,
        transform=prediction_transform,
        class_names=class_names,
    )

    if image_path.stem.startswith("cat"):
        actual_class = "cats"
    elif image_path.stem.startswith("dog"):
        actual_class = "dogs"
    else:
        continue

    is_correct = predicted_class == actual_class

    correct_predictions += int(is_correct)
    total_images += 1

    print(
        f"{image_path.name} | "
        f"Actual: {actual_class} | "
        f"Predicted: {predicted_class} | "
        f"Confidence: {confidence:.2%} | "
        f"Correct: {is_correct}"
    )

external_accuracy = correct_predictions / total_images

print()
print(f"Correct predictions: {correct_predictions}/{total_images}")
print(f"External images accuracy: {external_accuracy:.2%}")

cat-1.jpg | Actual: cats | Predicted: cats | Confidence: 80.21% | Correct: True
cat-10.jpg | Actual: cats | Predicted: dogs | Confidence: 55.82% | Correct: False
cat-11.jpg | Actual: cats | Predicted: cats | Confidence: 94.16% | Correct: True
cat-12.jpg | Actual: cats | Predicted: cats | Confidence: 68.95% | Correct: True
cat-13.jpg | Actual: cats | Predicted: dogs | Confidence: 70.07% | Correct: False
cat-14.jpg | Actual: cats | Predicted: cats | Confidence: 99.01% | Correct: True
cat-15.jpg | Actual: cats | Predicted: dogs | Confidence: 65.51% | Correct: False
cat-2.jpg | Actual: cats | Predicted: cats | Confidence: 79.01% | Correct: True
cat-3.jpg | Actual: cats | Predicted: dogs | Confidence: 93.29% | Correct: False
cat-4.jpg | Actual: cats | Predicted: cats | Confidence: 93.39% | Correct: True
cat-5.jpg | Actual: cats | Predicted: dogs | Confidence: 90.20% | Correct: False
cat-6.jpg | Actual: cats | Predicted: dogs | Confidence: 63.28% | Correct: False
cat-7.jpg | Actual: cats | P